## Peaks Calling
#### H3K27ac -- MACS2
#### H3K27me3 -- Epic2
#### H3K9me3 -- MACS2  --broad

In [ ]:
### env: chip_seq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool

In [2]:
chip_meta = pd.read_csv('chip_meta.csv',sep = ',')
chip_meta.head()

,id,Pol,Ac,Het,celltype,sex,age_group,age,PMI Hours,AgeGroup
0,390,NaN,390-GABA-ac,NaN,GABA,F,AGE_0,0.340000,18.0,Infancy
1,390,NaN,390-Glu-ac,NaN,Glu,F,AGE_0,0.340000,18.0,Infancy
2,4365,4365-GABA-pol,4365-GABA-ac,4365-GABA-het,GABA,M,AGE_0,0.030137,8.7,Infancy
3,4365,4365-Glu-pol,4365-Glu-ac,4365-Glu-het,Glu,M,AGE_0,0.030137,8.7,Infancy
4,4411,4411-GABA-pol,4411-GABA-ac,4411-GABA-het,GABA,M,AGE_0,0.150000,18.0,Infancy


In [2]:
input_meta = pd.read_csv('input_meta.csv')

## H3K27ac Peak Calling MACS2 

In [4]:
file_path = '/datasets/Public_Datasets/Dracheva_PsychEncode_development/processed/ChIP_GluGABA_filtered'
input_file_path = '/datasets/Public_Datasets/Dracheva_PsychEncode_development/raw/ChIPseq_bam_Aug2024/Input'

In [ ]:
def process_row(row):
    id = row['id']
    filename = row['Ac']
    celltype = row['celltype']
    agegroup = row['AgeGroup']

    bamfile = f'{file_path}/{filename}.normchr.map60_as30.bam'
    
    # Assuming input_meta is accessible and correctly defined
    input_id = input_meta[(input_meta['celltype'] == celltype) & (input_meta['AgeGroup'] == agegroup)]['id'].iloc[0]
    input_bam = f'{input_file_path}/INPUT-{input_id}-{celltype}.normchr.bam'

    peak_calling_command = f'macs2 callpeak -t {bamfile} \
                                            -c {input_bam} \
                                            -f BAMPE -g hs \
                                            -n {filename} \
                                            --outdir peaks/ac_peaks 2> peaks/ac_peaks/{filename}-macs2.log'
    os.system(peak_calling_command)  

# Filter and dropna as you did before
chip_meta_ac = chip_meta[['id', 'Ac', 'AgeGroup', 'celltype']].dropna()

pool = Pool(processes=14)  # Define the number of concurrent processes
pool.map(process_row, chip_meta_ac.to_dict('records'))
pool.close()
pool.join()

## Merge all samples

In [15]:
%%bash
dir="peaks/ac_peaks"
blacklist="hg38-blacklist.v2.nochr.bed"

for celltype in Glu GABA; do
    echo "Processing $celltype peaks..."

    # Step 1: Capture the list of files for the current celltype
    files=$(ls $dir/*${celltype}*peaks.narrowPeak)
    echo "Found files for $celltype: $files"

    # Step 2: Create a temporary directory for filtered narrowPeak files
    tmp_dir="/scratch/tmpHX/${celltype}_peaks"
    mkdir -p $tmp_dir
    echo "Created temporary directory: $tmp_dir"

    # Step 3: Filter each narrowPeak file based on -log10(q) > 3
    for file in $files; do
        #echo "Filtering $file..."
        awk '$9 > 3' $file > ${tmp_dir}/$(basename $file)
    done

    # Step 4: Run bedtools multiinter with the filtered files
    filtered_files=$(ls $tmp_dir/*${celltype}*peaks.narrowPeak)
    echo "Running bedtools multiinter on filtered files..."
    bedtools multiinter -i $filtered_files > ${dir}/multi_${celltype}_intersections.bed

    # Step 5: Filter regions overlapping in more than 3 samples and get union of contributing regions
    echo "Filtering regions overlapping in >= 3 samples..."
    awk '$4 >= 3' ${dir}/multi_${celltype}_intersections.bed > ${dir}/filtered_${celltype}_regions_tmp.bed

    echo "Extracting and merging contributing regions..."
    cat $filtered_files | \
    bedtools intersect -a stdin -b ${dir}/filtered_${celltype}_regions_tmp.bed -wa | \
    sort -k1,1 -k2,2n | \
    bedtools merge -i stdin > ${dir}/filtered_${celltype}_regions.bed

    # Step 6: Remove blacklisted regions
    echo "Removing blacklisted regions..."
    bedtools subtract -a ${dir}/filtered_${celltype}_regions.bed -b $blacklist -A > ${dir}/cleaned_${celltype}_regions.bed

    # Optional Step 7: Merge and extend cleaned regions
    # Uncomment if necessary
    echo "Merging and extending regions..."
    bedtools merge -i ${dir}/cleaned_${celltype}_regions.bed > ${dir}/${celltype}_ac_merged_new.bed

    # Step 8: Clean up temporary files and directory
    echo "Cleaning up temporary files..."
    rm -rf $tmp_dir
    rm -f ${dir}/multi_${celltype}_intersections.bed
    rm -f ${dir}/filtered_${celltype}_regions_tmp.bed
    rm -f ${dir}/filtered_${celltype}_regions.bed
    rm -f ${dir}/cleaned_${celltype}_regions.bed

    echo "$celltype processing complete."
done

Processing Glu peaks...
Found files for Glu: peaks/ac_peaks/1105-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/1275-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/1539-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/1823-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/390-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4332-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4337-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4365-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4369-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4379-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4411-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4414-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4428-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/4545-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/5077-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/5309-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/5570-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/5871-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/5936-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/5976-Glu-ac_peaks.narrowPeak
peaks/ac_peaks/5977-Glu-ac_peaks.narrowPeak
Created temporary directory: /sc

## Check celltype specific peaks

In [16]:
%%bash
dir="peaks/ac_peaks"

# Find GABA-specific peaks (present in GABA but not in Glu)
bedtools intersect -a ${dir}/GABA_ac_merged_new.bed -b ${dir}/Glu_ac_merged_new.bed -v > ${dir}/GABA_specific_ac.bed

# Find Glu-specific peaks (present in Glu but not in GABA)
bedtools intersect -a ${dir}/Glu_ac_merged_new.bed -b ${dir}/GABA_ac_merged_new.bed -v > ${dir}/Glu_specific_ac.bed

# Find common peaks, outputting regions from GABA that overlap with Glu
bedtools intersect -a ${dir}/GABA_ac_merged_new.bed -b ${dir}/Glu_ac_merged_new.bed -wa > ${dir}/tmp1.bed

# Find common peaks, outputting regions from Glu that overlap with GABA
bedtools intersect -a ${dir}/Glu_ac_merged_new.bed -b ${dir}/GABA_ac_merged_new.bed -wa > ${dir}/tmp2.bed

# Combine and sort common peaks from both GABA and Glu
cat ${dir}/tmp1.bed ${dir}/tmp2.bed | sort -k1,1 -k2,2n > ${dir}/tmp3.bed

# Merge sorted peaks into unified common regions
bedtools merge -i ${dir}/tmp3.bed > ${dir}/GABA_Glu_ac_common.bed

# Clean up temporary files
rm ${dir}/tmp1.bed ${dir}/tmp2.bed ${dir}/tmp3.bed

## H3K27me3 Peak Calling Epic2

In [7]:
def process_row(row):
    id = row['id']
    filename = row['Pol']
    celltype = row['celltype']
    agegroup = row['AgeGroup']

    bamfile = f'{file_path}/{filename}.normchr.map60_as30.bam'
    
    # Assuming input_meta is accessible and correctly defined
    input_id = input_meta[(input_meta['celltype'] == celltype) & (input_meta['AgeGroup'] == agegroup)]['id'].iloc[0]
    input_bam = f'{input_file_path}/INPUT-{input_id}-{celltype}.normchr.bam'

    peak_calling_command = f'epic2 -t {bamfile} -c {input_bam} --t bedpe -gn hg38 --bin-size 200 --gaps-allowed 3 --autodetect-chroms --output peaks/pol_peaks/{filename}.epic2.peak'
                          
                                   
    os.system(peak_calling_command)  

#Filter and dropna as you did before
chip_meta_pol = chip_meta[['id', 'Pol', 'AgeGroup', 'celltype']].dropna()



pool = Pool(processes=10)  # Define the number of concurrent processes
pool.map(process_row, chip_meta_pol.to_dict('records'))
pool.close()
pool.join()

Found a median readlength of 100.0
 
Found a median readlength of 100.0
 
Chromosomes included in analysis: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, GL000008.2, GL000009.2, GL000194.1, GL000195.1, GL000205.2, GL000208.1, GL000213.1, GL000214.1, GL000216.2, GL000218.1, GL000219.1, GL000220.1, GL000221.1, GL000224.1, GL000225.1, GL000226.1, KI270302.1, KI270303.1, KI270304.1, KI270305.1, KI270310.1, KI270311.1, KI270312.1, KI270315.1, KI270316.1, KI270317.1, KI270320.1, KI270322.1, KI270329.1, KI270330.1, KI270333.1, KI270334.1, KI270335.1, KI270336.1, KI270337.1, KI270338.1, KI270340.1, KI270362.1, KI270363.1, KI270364.1, KI270366.1, KI270371.1, KI270372.1, KI270373.1, KI270374.1, KI270375.1, KI270376.1, KI270378.1, KI270379.1, KI270381.1, KI270382.1, KI270383.1, KI270384.1, KI270385.1, KI270386.1, KI270387.1, KI270388.1, KI270389.1, KI270390.1, KI270391.1, KI270392.1, KI270393.1, KI270394.1, KI270395.1, KI270396.1, KI270411.1, KI270412.1, KI270414.

In [23]:
%%bash
dir="peaks/pol_peaks"
blacklist="hg38-blacklist.v2.nochr.bed"

for celltype in Glu GABA; do
    echo "Processing $celltype peaks..."

    # Step 1: Capture the list of files for the current celltype
    files=$(ls $dir/*${celltype}*epic2.peak)
    echo "Found files for $celltype: $files"
    
    echo "Running bedtools multiinter on filtered files..."
    bedtools multiinter -i $files > ${dir}/multi_${celltype}_intersections.bed

    # Step 6: Filter regions overlapping in >= 3 samples
    echo "Filtering regions overlapping in >= 3 samples..."
    awk '$4 >= 3' ${dir}/multi_${celltype}_intersections.bed > ${dir}/filtered_${celltype}_regions_tmp.bed

    echo "Extracting and merging contributing regions..."
    cat $files | \
    bedtools intersect -a stdin -b ${dir}/filtered_${celltype}_regions_tmp.bed -wa | \
    sort -k1,1 -k2,2n | \
    bedtools merge -i stdin > ${dir}/filtered_${celltype}_regions.bed

    # Step 7: Remove blacklisted regions
    echo "Removing blacklisted regions..."
    bedtools subtract -a ${dir}/filtered_${celltype}_regions.bed -b $blacklist -A > ${dir}/cleaned_${celltype}_regions.bed

    echo "Merging and extending regions..."
    bedtools merge -i ${dir}/cleaned_${celltype}_regions.bed > ${dir}/${celltype}_pol_merged_new.bed
    # Step 8: Clean up temporary files and directory
    
    echo "Cleaning up temporary files..."

    rm -f ${dir}/multi_${celltype}_intersections.bed
    rm -f ${dir}/filtered_${celltype}_regions_tmp.bed
    rm -f ${dir}/filtered_${celltype}_regions.bed
    rm -f ${dir}/cleaned_${celltype}_regions.bed

    echo "$celltype processing complete."
done

Processing Glu peaks...
Found files for Glu: peaks/pol_peaks/1105-Glu-pol.epic2.peak
peaks/pol_peaks/1275-Glu-pol.epic2.peak
peaks/pol_peaks/1539-Glu-pol.epic2.peak
peaks/pol_peaks/1823-Glu-pol.epic2.peak
peaks/pol_peaks/1848-Glu-pol.epic2.peak
peaks/pol_peaks/4332-Glu-pol.epic2.peak
peaks/pol_peaks/4337-Glu-pol.epic2.peak
peaks/pol_peaks/4365-Glu-pol.epic2.peak
peaks/pol_peaks/4369-Glu-pol.epic2.peak
peaks/pol_peaks/4379-Glu-pol.epic2.peak
peaks/pol_peaks/4411-Glu-pol.epic2.peak
peaks/pol_peaks/4414-Glu-pol.epic2.peak
peaks/pol_peaks/4428-Glu-pol.epic2.peak
peaks/pol_peaks/4545-Glu-pol.epic2.peak
peaks/pol_peaks/5077-Glu-pol.epic2.peak
peaks/pol_peaks/5387-Glu-pol.epic2.peak
peaks/pol_peaks/5570-Glu-pol.epic2.peak
peaks/pol_peaks/5617-Glu-pol.epic2.peak
peaks/pol_peaks/5871-Glu-pol.epic2.peak
peaks/pol_peaks/5936-Glu-pol.epic2.peak
peaks/pol_peaks/5976-Glu-pol.epic2.peak
peaks/pol_peaks/5977-Glu-pol.epic2.peak
Running bedtools multiinter on filtered files...
Filtering regions overlapp

In [42]:
%%bash
dir="peaks/pol_peaks"
blacklist="hg38-blacklist.v2.nochr.bed"
# samples=(4545 1539 5570)  # Early_Adulthood
samples=(4365 4411 4414 4428)  # Infancy
age_group=Infancy

for celltype in Glu GABA; do
    echo "Processing $celltype peaks..."

    # Step 1: Capture the list of files for the current celltype
    file_list=()
    for sample in "${samples[@]}"; do
        file_list+=("$dir/${sample}-${celltype}*epic2.peak")
    done

    # Convert array to space-separated string
    files=$(printf "%s " "${file_list[@]}")

    echo "Running bedtools multiinter on filtered files..."
    bedtools multiinter -i $files > ${dir}/multi_${celltype}_intersections.bed

    # Step 6: Filter regions overlapping in >= 2 samples
    echo "Filtering regions overlapping in >= 2 samples..."
    awk '$4 >= 2' ${dir}/multi_${celltype}_intersections.bed > ${dir}/filtered_${celltype}_regions_tmp.bed

    echo "Extracting and merging contributing regions..."
    cat $files | \
    bedtools intersect -a stdin -b ${dir}/filtered_${celltype}_regions_tmp.bed -wa | \
    sort -k1,1 -k2,2n | \
    bedtools merge -i stdin > ${dir}/filtered_${celltype}_regions.bed

    # Step 7: Remove blacklisted regions
    echo "Removing blacklisted regions..."
    bedtools subtract -a ${dir}/filtered_${celltype}_regions.bed -b $blacklist -A > ${dir}/cleaned_${celltype}_regions.bed

    echo "Merging and extending regions..."
    bedtools merge -i ${dir}/cleaned_${celltype}_regions.bed > ${dir}/${celltype}_pol_${age_group}_new.bed
    # Step 8: Clean up temporary files and directory
    
    echo "Cleaning up temporary files..."

    rm -f ${dir}/multi_${celltype}_intersections.bed
    rm -f ${dir}/filtered_${celltype}_regions_tmp.bed
    rm -f ${dir}/filtered_${celltype}_regions.bed
    rm -f ${dir}/cleaned_${celltype}_regions.bed

    echo "$celltype processing complete."
done

Processing Glu peaks...
Running bedtools multiinter on filtered files...
Filtering regions overlapping in >= 2 samples...
Extracting and merging contributing regions...
Removing blacklisted regions...
Merging and extending regions...
Cleaning up temporary files...
Glu processing complete.
Processing GABA peaks...
Running bedtools multiinter on filtered files...
Filtering regions overlapping in >= 2 samples...
Extracting and merging contributing regions...
Removing blacklisted regions...
Merging and extending regions...
Cleaning up temporary files...
GABA processing complete.


## H3K9me3 Peak Calling MACS2 

In [12]:
def process_row(row):
    id = row['id']
    filename = row['Het']
    celltype = row['celltype']
    agegroup = row['AgeGroup']

    bamfile = f'{file_path}/{filename}.normchr.map60_as30.bam'
    
    # Assuming input_meta is accessible and correctly defined
    input_id = input_meta[(input_meta['celltype'] == celltype) & (input_meta['AgeGroup'] == agegroup)]['id'].iloc[0]
    input_bam = f'{input_file_path}/INPUT-{input_id}-{celltype}.normchr.bam'

    peak_calling_command = f'macs2 callpeak -t {bamfile} \
                                            -c {input_bam} \
                                            -f BAM -g hs \
                                            -n {filename} \
                                            --broad \
                                            --broad-cutoff 0.01 -q 0.01 \
                                            --outdir peaks/het_peaks 2 > peaks/het_peaks/{filename}-macs2.log'
    os.system(peak_calling_command)  

# Filter and dropna as you did before
chip_meta_het = chip_meta[['id', 'Het', 'AgeGroup', 'celltype']].dropna()



pool = Pool(processes=12)  # Define the number of concurrent processes
pool.map(process_row, chip_meta_het.to_dict('records'))
pool.close()
pool.join()

In [25]:
%%bash
dir="peaks/het_peaks"
blacklist="hg38-blacklist.v2.nochr.bed"
for celltype in GABA Glu; do
    # Step 1: Capture the list of files for the current celltype
    files=$(ls ${dir}/*${celltype}*broadPeak)

    # Step 2: Create a temporary directory in /scratch for filtered Peak files
    # tmp_dir="/scratch/tmpHX/${celltype}_peaks"
    # mkdir -p $tmp_dir

    
    # for file in $files; do
    #     awk '$5 > 10' $file > ${tmp_dir}/$(basename $file)
    # done
    
    echo "Running bedtools multiinter on filtered files..."
    bedtools multiinter -i $files > ${dir}/multi_${celltype}_intersections.bed

    # Step 6: Filter regions overlapping in >= 3 samples
    echo "Filtering regions overlapping in >= 3 samples..."
    awk '$4 >= 3' ${dir}/multi_${celltype}_intersections.bed > ${dir}/filtered_${celltype}_regions_tmp.bed

    echo "Extracting and merging contributing regions..."
    cat $files | \
    bedtools intersect -a stdin -b ${dir}/filtered_${celltype}_regions_tmp.bed -wa | \
    sort -k1,1 -k2,2n | \
    bedtools merge -i stdin > ${dir}/filtered_${celltype}_regions.bed

    # Step 7: Remove blacklisted regions
    echo "Removing blacklisted regions..."
    bedtools subtract -a ${dir}/filtered_${celltype}_regions.bed -b $blacklist -A > ${dir}/cleaned_${celltype}_regions.bed

    echo "Merging and extending regions..."
    bedtools merge -i ${dir}/cleaned_${celltype}_regions.bed > ${dir}/${celltype}_het_merged_new.bed
    # Step 8: Clean up temporary files and directory
    
    echo "Cleaning up temporary files..."

    rm -f ${dir}/multi_${celltype}_intersections.bed
    rm -f ${dir}/filtered_${celltype}_regions_tmp.bed
    rm -f ${dir}/filtered_${celltype}_regions.bed
    rm -f ${dir}/cleaned_${celltype}_regions.bed

    echo "$celltype processing complete."
done

Running bedtools multiinter on filtered files...
Filtering regions overlapping in >= 3 samples...
Extracting and merging contributing regions...
Removing blacklisted regions...
Merging and extending regions...
Cleaning up temporary files...
GABA processing complete.
Running bedtools multiinter on filtered files...
Filtering regions overlapping in >= 3 samples...
Extracting and merging contributing regions...
Removing blacklisted regions...
Merging and extending regions...
Cleaning up temporary files...
Glu processing complete.


### Create Diffbind files to extract counts

In [26]:
file_path = '/datasets/Public_Datasets/Dracheva_PsychEncode_development/processed/ChIP_GluGABA_filtered'

In [29]:
def generate_csv(mod_type):
    chip_meta_sub = chip_meta[['id', mod_type, 'celltype', 'AgeGroup', 'age', 'PMI Hours']].dropna()
    #chip_meta_sub_out = []
    for celltype in ['GABA', 'Glu']:
        chip_meta_sub_filtered = chip_meta_sub[chip_meta_sub['celltype'] == celltype][['id', 'AgeGroup', 'age', 'PMI Hours']]
        chip_meta_sub_filtered.columns = ['SampleID', 'Condition', 'age', 'PMI']
        chip_meta_sub_filtered['age'] = round(chip_meta_sub_filtered['age'], 3)
        chip_meta_sub_filtered['Tissue'] = celltype
        chip_meta_sub_filtered['Replicates'] =  chip_meta_sub_filtered.groupby('Condition').cumcount() + 1
        chip_meta_sub_filtered['bamReads'] = [f'{file_path}/paired/{subject}-{celltype}-{mod_type.lower()}.normchr.map60_as30_paired.bam' for subject in chip_meta_sub_filtered['SampleID']]

        chip_meta_sub_filtered['Peaks'] = f'/cndd/hex002/PsychEncode/ChIP/peaks/{mod_type.lower()}_peaks/{celltype}_{mod_type.lower()}_merged_new.bed'
        
        chip_meta_sub_filtered['PeakCaller'] = 'macs2'
        #print(chip_meta_sub_filtered)
        chip_meta_sub_filtered.to_csv(f'peaks/Sample_sheet_{mod_type}_{celltype}_peaks.csv', sep=',',index = False)
    

In [30]:
generate_csv('Ac')

In [31]:
generate_csv('Pol')

In [32]:
generate_csv('Het')